# CRIVO — Experimento Pointer-Copy Causal (GPU)

Experimento **cirúrgico**: Transformer e lm_head ficam congelados. A única parte treinável é uma cabeça low-rank de pointer/copy + gate geração↔cópia.

Protocolo: 1.000 passos, 100 entidades cegas, 600 casos finais (100 cópia pura + 500 semânticos), nomes 1–4 tokens, grupos lexicais close/distant e definição no início/meio/fim. Nada é ativado no chat.

Hotfix: train/dev/blind são forçados a ser disjuntos sem prefixos específicos por split.


In [ ]:
!nvidia-smi
!pip -q install 'tokenizers>=0.20,<1'
import torch
from google.colab import drive
assert torch.cuda.is_available(), 'Ative GPU em Ambiente de execução > Alterar tipo de ambiente > GPU.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf CRIVO
!git clone -q --branch exp/pointer-copy-causal-20261012 --single-branch https://github.com/HROSONE/CRIVO.git
%cd /content/CRIVO
!git rev-parse HEAD


## Preparar checkpoint-base no SSD local

Copia somente a fase 5 preservada para `/content`, evitando ler os pesos pela montagem do Drive durante o experimento.


In [ ]:
import os, shutil, subprocess, sys, signal, time
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CRIVO/checkpoints')
LOCAL_ROOT = Path('/content/crivo_pointer_root')

subprocess.run(['bash','-lc', "pkill -f '[e]xperimento_pointer_copy' || true"], check=False)
if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)

curricular = DRIVE_ROOT/'escola_curricular_longa'
runs = sorted(p for p in curricular.glob('run_*') if (p/'05_05_dialogo_estado'/'pesos.pt').exists())
assert runs, 'Não encontrei a fase 5 do treino curricular no Drive.'
src = runs[-1]/'05_05_dialogo_estado'
dst = LOCAL_ROOT/'escola_curricular_longa'/runs[-1].name/'05_05_dialogo_estado'
dst.mkdir(parents=True, exist_ok=True)
print('BASE:', src)
for nome in ('pesos.pt','tokenizer.json'):
    print('Copiando', nome, 'para SSD local...', flush=True)
    shutil.copy2(src/nome, dst/nome)
print('✅ Base local pronta:', dst)


## Rodar experimento

A saída aparece ao vivo. Primeiro ele valida os splits disjuntos, mede o **baseline congelado**, treina somente o pointer/gate e mede exatamente o mesmo holdout.


In [ ]:
env=os.environ.copy()
env['PYTHONPATH']='/content/CRIVO'
env['PYTHONHASHSEED']='0'
env['PYTHONUNBUFFERED']='1'
cmd=[sys.executable,'-u','/content/CRIVO/scripts/experimento_pointer_copy_cli.py',
     '--drive-root',str(LOCAL_ROOT),'--batch-size','20','--steps','1000']
print('🚀 Iniciando pointer-copy causal...')
p=subprocess.Popen(cmd,cwd='/content/CRIVO',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in p.stdout:
        print(line,end='',flush=True)
    rc=p.wait()
except KeyboardInterrupt:
    print('\nInterrompido; encerrando processo filho...')
    try: os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except Exception: p.terminate()
    raise
assert rc == 0, f'Experimento falhou com código {rc}'


## Salvar resultado no Drive


In [ ]:
local_out=LOCAL_ROOT/'experimento_pointer_copy'
drive_out=DRIVE_ROOT/'experimento_pointer_copy'
assert local_out.exists(), 'Não encontrei a saída local.'
drive_out.mkdir(parents=True,exist_ok=True)
shutil.copytree(local_out,drive_out,dirs_exist_ok=True)
print('✅ RESULTADOS SALVOS:',drive_out)
print('Procure no final por: EXPERIMENTO POINTER COPY FINALIZADO')
print('ativo_no_chat=false')
